<a href="https://colab.research.google.com/github/opeyemiolawuwo18-droid/QPI-net/blob/main/QPI_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pytorch-msssim

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from google.colab import drive
from pytorch_msssim import SSIM

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Device Check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 3. Dataset Blueprint
class QPIDataset(Dataset):
    def __init__(self, dataset_dir):
        self.dataset_dir = dataset_dir
        self.inputs_dir = os.path.join(dataset_dir, 'inputs')
        self.targets_dir = os.path.join(dataset_dir, 'targets')
        self.file_list = sorted(os.listdir(self.inputs_dir))

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        input_arr = np.load(os.path.join(self.inputs_dir, file_name))
        target_arr = np.load(os.path.join(self.targets_dir, file_name))

        input_tensor = torch.from_numpy(input_arr).float()
        target_tensor = torch.from_numpy(target_arr).float().unsqueeze(0)
        return input_tensor, target_tensor

# 4. Model Architecture Blocks
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class QPINetUNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1):
        super(QPINetUNet, self).__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.bottleneck = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))

        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(256, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.bottleneck(x3)

        x = self.up1(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.conv_up1(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv_up2(x)

        x = self.up3(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv_up3(x)
        return self.outc(x)

# 5. Load Dataset and DataLoaders
dataset_dir = '/content/drive/MyDrive/QPI-net/qpi_module2/dataset/'
full_dataset = QPIDataset(dataset_dir)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

Mounted at /content/drive
Using device: cuda


In [ ]:
class ASMWavePropagator(nn.Module):
    def __init__(self, wavelength=532e-9, dx=0.325e-6, shape=(512, 512)):
        super(ASMWavePropagator, self).__init__()
        self.wavelength = wavelength
        self.dx = dx
        self.k = 2 * torch.pi / wavelength

        Ny, Nx = shape
        fx = torch.fft.fftfreq(Nx, d=dx)
        fy = torch.fft.fftfreq(Ny, d=dx)
        FX, FY = torch.meshgrid(fx, fy, indexing='ij')

        arg = 1.0 - (wavelength * FX)**2 - (wavelength * FY)**2
        arg = torch.clamp(arg, min=0.0)
        kz = self.k * torch.sqrt(arg)

        self.register_buffer('kz', kz)

    def forward(self, phase, z_dist):
        u0 = torch.complex(torch.cos(phase), torch.sin(phase))
        U0 = torch.fft.fft2(u0)
        H = torch.exp(1j * self.kz * z_dist)
        Uz = torch.fft.ifft2(U0 * H)
        return torch.abs(Uz)**2

class QPIPhysicsHybridLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.25, gamma=0.1, z_dist=50e-6):
        super(QPIPhysicsHybridLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.z_dist = z_dist

        self.mse = nn.MSELoss()
        self.l1 = nn.L1Loss()
        self.ssim_loss = SSIM(data_range=5.0, channel=1, spatial_dims=2)
        self.asm_propagator = ASMWavePropagator(shape=(512, 512))

        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def _gradient_loss(self, pred, target):
        pred_grad_x = F.conv2d(pred, self.sobel_x, padding=1)
        pred_grad_y = F.conv2d(pred, self.sobel_y, padding=1)
        target_grad_x = F.conv2d(target, self.sobel_x, padding=1)
        target_grad_y = F.conv2d(target, self.sobel_y, padding=1)
        return F.l1_loss(pred_grad_x, target_grad_x) + F.l1_loss(pred_grad_y, target_grad_y)

    def forward(self, pred_phase, target_phase, inputs_defocus):
        loss_mse = self.mse(pred_phase, target_phase)
        loss_ssim = 1.0 - self.ssim_loss(pred_phase, target_phase)
        loss_grad = self._gradient_loss(pred_phase, target_phase)

        i_pos_target = inputs_defocus[:, 0:1, :, :]
        i_neg_target = inputs_defocus[:, 1:2, :, :]

        i_pos_sim = self.asm_propagator(pred_phase, +self.z_dist)
        i_neg_sim = self.asm_propagator(pred_phase, -self.z_dist)

        loss_physics = self.l1(i_pos_sim, i_pos_target) + self.l1(i_neg_sim, i_neg_target)

        total_loss = loss_mse + (self.alpha * loss_ssim) + (self.beta * loss_grad) + (self.gamma * loss_physics)
        return total_loss, loss_mse, loss_physics

In [ ]:

# Model Training Loop (30 Epochs)

import os
import torch
import torch.nn as nn

# 1. Instantiate Model & Set Paths
model = QPINetUNet(in_channels=2, out_channels=1).to(device)
weights_path = '/content/drive/MyDrive/QPI-net/qpi_module2/qpi_net_robust_weights.pth'

# 2. Loss Functions & Optimizer
l1_loss = nn.L1Loss().to(device)
mse_loss = nn.MSELoss().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

# 3. Training Loop Execution
num_epochs = 30
print("🚀 Launching Baseline 30-Epoch QPI-net Training...\n", flush=True)

best_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        # Combined L1 + 0.5*MSE loss
        loss = l1_loss(outputs, targets) + (0.5 * mse_loss(outputs, targets))

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)

    scheduler.step()
    epoch_loss = train_loss / len(train_dataset)

    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | Loss: {epoch_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.6f}", flush=True)

    # Save best performing weights to Google Drive
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        os.makedirs(os.path.dirname(weights_path), exist_ok=True)
        torch.save(model.state_dict(), weights_path)

print(f"\n💾 Training complete! Checkpoint saved to: {weights_path}")

🚀 Launching Baseline 30-Epoch QPI-net Training...

Epoch [01/30] | Loss: 0.911723 | LR: 0.000997
Epoch [02/30] | Loss: 0.496605 | LR: 0.000989
Epoch [03/30] | Loss: 0.472454 | LR: 0.000976
Epoch [04/30] | Loss: 0.432741 | LR: 0.000957
Epoch [05/30] | Loss: 0.422650 | LR: 0.000933


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# 1. Load sample for testing
sample_name = sorted(os.listdir('/content/drive/MyDrive/QPI-net/qpi_module2/dataset/inputs'))[0]

input_sample = np.load(os.path.join('/content/drive/MyDrive/QPI-net/qpi_module2/dataset/inputs', sample_name))
target_sample = np.load(os.path.join('/content/drive/MyDrive/QPI-net/qpi_module2/dataset/targets', sample_name))

# 2. Model Inference
input_tensor = torch.from_numpy(input_sample).float().unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    pinn_pred_tensor = model(input_tensor)

gt_phase = target_sample
pinn_pred_phase = pinn_pred_tensor.squeeze().cpu().numpy()

# 3. Calculate PSNR & SSIM
psnr_pinn = psnr(gt_phase, pinn_pred_phase, data_range=gt_phase.max() - gt_phase.min())
ssim_pinn = ssim(gt_phase, pinn_pred_phase, data_range=gt_phase.max() - gt_phase.min())

print("📊 PHYSICS-INFORMED (PINN) PERFORMANCE METRICS:")
print(f" ├── PSNR: {psnr_pinn:.2f} dB")
print(f" └── SSIM: {ssim_pinn:.4f}")

# 4. Visual Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(input_sample[0], cmap='gray')
axes[0].set_title("Input Defocus Intensity ($I_{pos}$)")

axes[1].imshow(gt_phase, cmap='viridis')
axes[1].set_title("Ground Truth Phase ($\phi$) [rad]")

im2 = axes[2].imshow(pinn_pred_phase, cmap='viridis')
axes[2].set_title("PINN Predicted Phase ($\hat{\phi}$) [rad]")
plt.colorbar(im2, ax=axes[2])

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()